In [ ]:
!pip install pybaseball pandas pymongo dnspython

In [2]:
import pandas as pd
from pybaseball import statcast_pitcher_expected_stats, statcast_pitcher_pitch_arsenal

In [3]:
def get_pitcher_data_for_season(year):
    expected_df = statcast_pitcher_expected_stats(year, minPA=1)
    speed_df = statcast_pitcher_pitch_arsenal(year, minP=1, arsenal_type="avg_speed")
    spin_df = statcast_pitcher_pitch_arsenal(year, minP=1, arsenal_type="avg_spin")

    return expected_df, speed_df, spin_df

In [4]:
def clean_and_merge_season(year):
    expected_df, speed_df, spin_df = get_pitcher_data_for_season(year)

    expected_keep = expected_df[[
        "player_id",
        "last_name, first_name",
        "year",
        "xera"
    ]].copy()

    expected_keep = expected_keep.rename(columns={
        "player_id": "pitcher_id",
        "last_name, first_name": "pitcher_name",
        "year": "season",
        "xera": "xERA"
    })

    optional_feature_map = {
        "exit_velocity_avg": "avg_launch_speed",
        "launch_angle_avg": "avg_launch_angle",
        "estimated_woba_using_speedangle": "avg_estimated_woba",
        "est_woba": "avg_estimated_woba",
        "pa": "plate_appearances",
        "pitches": "pitch_count",
        "k_percent": "strikeout_rate",
        "bb_percent": "walk_rate"
    }

    for old_col, new_col in optional_feature_map.items():
        if old_col in expected_df.columns:
            expected_keep[new_col] = expected_df[old_col]

    speed_keep = speed_df[[
        "pitcher",
        "last_name, first_name",
        "ff_avg_speed"
    ]].copy()

    speed_keep = speed_keep.rename(columns={
        "pitcher": "pitcher_id",
        "last_name, first_name": "pitcher_name_speed",
        "ff_avg_speed": "fastball_velocity"
    })

    spin_keep = spin_df[[
        "pitcher",
        "last_name, first_name",
        "ff_avg_spin"
    ]].copy()

    spin_keep = spin_keep.rename(columns={
        "pitcher": "pitcher_id",
        "last_name, first_name": "pitcher_name_spin",
        "ff_avg_spin": "fastball_spin"
    })

    merged = expected_keep.merge(
        speed_keep[["pitcher_id", "fastball_velocity"]],
        on="pitcher_id",
        how="left"
    )

    merged = merged.merge(
        spin_keep[["pitcher_id", "fastball_spin"]],
        on="pitcher_id",
        how="left"
    )

    numeric_columns = [
        "xERA",
        "fastball_velocity",
        "fastball_spin",
        "avg_launch_speed",
        "avg_launch_angle",
        "avg_estimated_woba",
        "plate_appearances",
        "pitch_count",
        "strikeout_rate",
        "walk_rate"
    ]

    for col in numeric_columns:
        if col in merged.columns:
            merged[col] = pd.to_numeric(merged[col], errors="coerce")

    required_columns = [
        "xERA",
        "fastball_velocity",
        "fastball_spin"
    ]

    merged = merged.dropna(subset=required_columns)

    merged = merged.drop_duplicates(
        subset=["pitcher_id", "season"]
    ).reset_index(drop=True)

    return merged, expected_df, speed_df, spin_df

In [6]:
years = [2018, 2019, 2020, 2021]

final_dfs = []
raw_expected_all = []
raw_speed_all = []
raw_spin_all = []

for year in years:
    merged_season, expected_raw, speed_raw, spin_raw = clean_and_merge_season(year)

    final_dfs.append(merged_season)

    expected_raw = expected_raw.copy()
    speed_raw = speed_raw.copy()
    spin_raw = spin_raw.copy()

    expected_raw["source_season"] = year
    speed_raw["source_season"] = year
    spin_raw["source_season"] = year

    raw_expected_all.append(expected_raw)
    raw_speed_all.append(speed_raw)
    raw_spin_all.append(spin_raw)

In [7]:
pitcher_season_df = pd.concat(final_dfs, ignore_index=True)

expected_raw_df = pd.concat(raw_expected_all, ignore_index=True)
speed_raw_df = pd.concat(raw_speed_all, ignore_index=True)
spin_raw_df = pd.concat(raw_spin_all, ignore_index=True)

In [8]:
print("Final merged dataset shape:", pitcher_season_df.shape)
print("\nMerged dataset columns:")
print(pitcher_season_df.columns.tolist())

print("\nPreview of merged dataset:")
display(pitcher_season_df.head())

print("\nRaw expected stats shape:", expected_raw_df.shape)
print("Raw speed shape:", speed_raw_df.shape)
print("Raw spin shape:", spin_raw_df.shape)

print("\nTotal raw documents if uploaded separately:")
print(len(expected_raw_df) + len(speed_raw_df) + len(spin_raw_df))

Final merged dataset shape: (2967, 8)

Merged dataset columns:
['pitcher_id', 'pitcher_name', 'season', 'xERA', 'avg_estimated_woba', 'plate_appearances', 'fastball_velocity', 'fastball_spin']

Preview of merged dataset:


,pitcher_id,pitcher_name,season,xERA,avg_estimated_woba,plate_appearances,fastball_velocity,fastball_spin
0,572971,"Keuchel, Dallas",2018,3.60,0.293,874,89.7,2166.0
1,448306,"Shields, James",2018,5.33,0.350,871,89.5,2271.0
2,453286,"Scherzer, Max",2018,2.56,0.248,866,94.4,2487.0
3,607536,"Freeland, Kyle",2018,3.76,0.299,844,91.8,2267.0
4,446372,"Kluber, Corey",2018,3.18,0.276,842,92.0,2410.0



Raw expected stats shape: (3274, 18)
Raw speed shape: (3166, 13)
Raw spin shape: (3166, 13)

Total raw documents if uploaded separately:
9606


In [9]:
pitcher_season_df.to_csv("pitcher_season_statcast_2018_2021.csv", index=False)
expected_raw_df.to_csv("pitcher_expected_stats_raw_2018_2021.csv", index=False)
speed_raw_df.to_csv("pitcher_pitch_arsenal_speed_raw_2018_2021.csv", index=False)
spin_raw_df.to_csv("pitcher_pitch_arsenal_spin_raw_2018_2021.csv", index=False)

print("CSV files saved.")

CSV files saved.


In [10]:
model_docs = pitcher_season_df.to_dict(orient="records")
expected_docs = expected_raw_df.to_dict(orient="records")
speed_docs = speed_raw_df.to_dict(orient="records")
spin_docs = spin_raw_df.to_dict(orient="records")

print("Sample model document:")
print(model_docs[0] if len(model_docs) > 0 else "No documents created.")

Sample model document:
{'pitcher_id': 572971, 'pitcher_name': 'Keuchel, Dallas', 'season': 2018, 'xERA': 3.6, 'avg_estimated_woba': 0.293, 'plate_appearances': 874, 'fastball_velocity': 89.7, 'fastball_spin': 2166.0}


In [11]:
model_collection = db["pitcher_season_model_data"]
expected_collection = db["pitcher_expected_stats_raw"]
speed_collection = db["pitcher_pitch_arsenal_speed_raw"]
spin_collection = db["pitcher_pitch_arsenal_spin_raw"]

In [12]:
model_collection.delete_many({})
expected_collection.delete_many({})
speed_collection.delete_many({})
spin_collection.delete_many({})

print("Old data cleared.")

Old data cleared.


In [13]:
if model_docs:
    model_collection.insert_many(model_docs)
if expected_docs:
    expected_collection.insert_many(expected_docs)
if speed_docs:
    speed_collection.insert_many(speed_docs)
if spin_docs:
    spin_collection.insert_many(spin_docs)

print("MongoDB upload complete.")

MongoDB upload complete.


In [14]:
print("Model docs:", model_collection.count_documents({}))
print("Expected raw docs:", expected_collection.count_documents({}))
print("Speed raw docs:", speed_collection.count_documents({}))
print("Spin raw docs:", spin_collection.count_documents({}))

print("Total docs:",
      model_collection.count_documents({}) +
      expected_collection.count_documents({}) +
      speed_collection.count_documents({}) +
      spin_collection.count_documents({}))

Model docs: 2967
Expected raw docs: 3274
Speed raw docs: 3166
Spin raw docs: 3166
Total docs: 12573


In [15]:
pitcher_season_df.info()
display(pitcher_season_df.head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2967 entries, 0 to 2966
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   pitcher_id          2967 non-null   int64  
 1   pitcher_name        2967 non-null   object 
 2   season              2967 non-null   int64  
 3   xERA                2967 non-null   float64
 4   avg_estimated_woba  2967 non-null   float64
 5   plate_appearances   2967 non-null   int64  
 6   fastball_velocity   2967 non-null   float64
 7   fastball_spin       2967 non-null   float64
dtypes: float64(4), int64(3), object(1)
memory usage: 185.6+ KB


,pitcher_id,pitcher_name,season,xERA,avg_estimated_woba,plate_appearances,fastball_velocity,fastball_spin
0,572971,"Keuchel, Dallas",2018,3.60,0.293,874,89.7,2166.0
1,448306,"Shields, James",2018,5.33,0.350,871,89.5,2271.0
2,453286,"Scherzer, Max",2018,2.56,0.248,866,94.4,2487.0
3,607536,"Freeland, Kyle",2018,3.76,0.299,844,91.8,2267.0
4,446372,"Kluber, Corey",2018,3.18,0.276,842,92.0,2410.0
5,425844,"Greinke, Zack",2018,3.87,0.303,839,89.5,2335.0
6,594798,"deGrom, Jacob",2018,2.46,0.243,835,96.0,2362.0
7,434378,"Verlander, Justin",2018,2.34,0.237,833,95.0,2618.0
8,605400,"Nola, Aaron",2018,2.77,0.258,831,92.7,2116.0
9,502043,"Gibson, Kyle",2018,4.33,0.319,826,93.1,2219.0


In [16]:
dupes = pitcher_season_df.duplicated(subset=["pitcher_id", "season"]).sum()
print("Duplicate pitcher-season rows:", dupes)

Duplicate pitcher-season rows: 0


In [17]:
print(pitcher_season_df[["xERA", "fastball_velocity", "fastball_spin"]].isna().sum())

xERA                 0
fastball_velocity    0
fastball_spin        0
dtype: int64
